# Data synthesis (데이터 합성)

Goal:anchor, correct reasoning, incorrect reasoning
(question)->correct reasoning, incorrect reasoning을 triplet으로 만듬

In [5]:
#train data 불러오기
import pandas as pd

df_train = pd.read_csv("../00_data/splits/medqa/train.csv")

In [6]:
#llama api 설정
import requests

URL = "http://10.189.26.12:30002/v1/chat/completions"

headers = {
    "Content-Type": "application/json",
}

In [7]:
#triplet 생성 함수
def generate_triplet(question, answer):

    prompt = f"""
You are a board-certified physician.

Question:
{question}

Correct Answer:
{answer}

Task:
1. Write a clinically correct reasoning explaining the answer.
2. Write an incorrect but plausible reasoning that could mislead a weaker model.

Constraints:
- Both should sound medically realistic
- Do NOT say which is correct or incorrect
- Keep it concise

Format:

CORRECT:
...

INCORRECT:
...
"""

    payload = {
        "model": "llama-3-3-70b-chat",
        "messages": [
            {"role": "system", "content": "You are a medical expert."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7,
        "max_tokens": 300
    }

    response = requests.post(URL, headers=headers, json=payload)

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print("Error:", response.text)
        return None

In [4]:
# test(핵심)
from tqdm import tqdm

triplets = []

for i in tqdm(range(10)):
    row = df_train.iloc[i]

    t = generate_triplet(row["question"], row["answer"])

    triplets.append({
        "question": row["question"],
        "answer": row["answer"],
        "triplet": t
    })

100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [00:36<00:00,  3.67s/it]


In [5]:
#결과 확인
for t in triplets[:2]:
    print(t["triplet"])
    print("="*50)

CORRECT:
The patient's recurrent Staphylococcus aureus infections and the clear nitroblue tetrazolium test result suggest a diagnosis of Chronic Granulomatous Disease (CGD). This condition is characterized by the inability of phagocytic cells to generate the microbicidal respiratory burst, which is essential for killing certain bacteria, including Staphylococcus aureus. The nitroblue tetrazolium test measures the production of superoxides, a key component of the respiratory burst, and a clear result indicates impaired superoxide production.

INCORRECT:
The patient's symptoms and test results are consistent with a diagnosis of leukocyte adhesion deficiency, where the inability of leukocytes to migrate to the site of infection leads to recurrent infections. The clear nitroblue tetrazolium test result may be a secondary effect of the underlying adhesion defect, which impairs the normal functioning of phagocytic cells and leads to an accumulation of bacteria, including Staphylococcus aureu

In [6]:
import json
import os

save_path = "../04_synthesis"
os.makedirs(save_path, exist_ok=True)

with open(f"{save_path}/medqa_triplets_llama_sample.json", "w") as f:
    json.dump(triplets, f, indent=2)

In [ ]:
#전체 test
import os
import json
from tqdm import tqdm

SAVE_PATH = "../04_synthesis/medqa_triplets_llama_full.json"

# 1️⃣ 기존 파일 있으면 불러오기
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, "r") as f:
        triplets = json.load(f)
    print(f"Resuming from {len(triplets)} samples")
else:
    triplets = []
    print("Starting from scratch")

# 2️⃣ 시작 index 자동 설정
start = len(triplets)
end = len(df_train)

# 3️⃣ loop
for i in tqdm(range(start, end)):
    row = df_train.iloc[i]

    t = generate_triplet(row["question"], row["answer"])

    if t is not None:
        triplets.append({
            "question": row["question"],
            "answer": row["answer"],
            "triplet": t
        })

    # 50개마다 자동 저장
    if i % 50 == 0:
        with open(SAVE_PATH, "w") as f:
            json.dump(triplets, f, indent=2)

# 4️⃣ 마지막 저장
with open(SAVE_PATH, "w") as f:
    json.dump(triplets, f, indent=2)

print("Done")

Resuming from 4501 samples


  0%|▍                                                                              | 13/2623 [00:57<3:27:56,  4.78s/it]